rm -rf /content/green-european-crab-1 --
rm -rf /content/crabs --
rm -rf /content/runs

In [1]:
# Install the Ultralytics package for YOLOv8
!pip install ultralytics

from ultralytics import YOLO
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mkJZ8KCgI4U4oHIjCWyB")
project = rf.workspace("crab-q5y75").project("green-european-crab-ohist")
version = project.version(1)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 52.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.90
    Uninstalling opencv-python-headless-4.13.0.90:
      Successfully uninstalled opencv-python-headless-4.13.0.90
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to green-european-crab-1 in yolov8:: 100%|██████████| 228/228 [00:00<00:00, 8147.61it/s]


In [3]:
print(os.getcwd())

/content


In [4]:
data = '/content/green-european-crab-1/data.yaml'

In [5]:
model = YOLO("yolov8n.pt")

In [ ]:
train_results = model.train(
    data=data,  # Path to dataset configuration file
    epochs=50,  # Number of training epochs
    imgsz=640,  # Image size for training
    device="0",  # Device to run on (e.g., 'cpu', 0, [0,1,2,3])
)

Ultralytics 8.4.12 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/green-european-crab-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pers

In [ ]:
metrics = model.val()

In [ ]:
model = YOLO('/content/runs/detect/train/weights/best.pt')

In [ ]:
# 'save=True' will still save the images with boxes drawn on them
results = model.predict(source='/content/green-european-crab-1/test/images', save=True, conf=0.6)

In [ ]:
data = '/content/green-european-crab-1/data.yaml'

In [ ]:
import cv2

# 1. Load your best model
model = YOLO('/content/runs/detect/train/weights/best.pt')

# 2. Run prediction on the folder
# We set save=False because we are handling the saving manually below
results = model.predict(source='/content/green-european-crab-1/test/images', conf=0.25)

print("Starting manual save process...")

# 3. Loop through EVERY result
for result in results:
    # Get the base filename (e.g., 'crab_1.jpg')
    image_path = result.path
    image_file = os.path.basename(image_path)

    # Get the image with bounding boxes (this is your 'filtered_frame')
    # It is already in BGR format, which cv2.imwrite likes
    filtered_frame = result.plot()

    # Get the count of detected crabs
    invasive_count = len(result.boxes)

    # Add the text to the image before saving
    text = f"Invasive count: {invasive_count}"
    # Position: (50, 100), Red color in BGR is (0, 0, 255)
    cv2.putText(filtered_frame, text, (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)

    # 4. Create the output filename and save (your specific code part)
    output_filename = f"output_{os.path.splitext(image_file)[0]}.jpg"
    cv2.imwrite(output_filename, filtered_frame)

    print(f"Detection complete for {image_file}. Invasive count: {invasive_count}. Image saved as {output_filename}")